# Pre Processing Dataset Baru

In [1]:
import pandas as pd
import nltk
import time
import re
import tqdm

nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## Load Data

In [2]:
df = pd.read_json('/content/drive/MyDrive/NER nlp/Dataset/Perdata Melawan Hukum 200.json')
df

,text,label
0,"[putusan, nomor, 698, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
1,"[penetapan, nomor, 306, /, pdt, ., g, /, 2022,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,"[putusan, nomor, 643, /, pdt, ., g, /, 2022, /...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,"[putusan, nomor, 534, /, pdt, ., g, /, 2020, /...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,"[putusan, nomor, 818, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
...,...,...
195,"[penetapan, nomor, ., 787, /, pdt, ., g, /, 20...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
196,"[putusan, sela, nomor, 190, /, pdt, ., g, /, 2...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
197,"[putusan, nomor, 52, /, pdt, g, /, 2023, /, pn...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
198,"[penetapan, nomor, 680, /, pdt, ., g, /, 2023,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."


### Menampilkan Text unik pada text-tags

In [ ]:
labels = [label for label in df['label'].values.tolist()]

unique_labels = set()

for lb in labels:
  [unique_labels.add(i) for i in lb if i not in unique_labels]

print(sorted(unique_labels))

['B_DENN', 'B_JUDP', 'B_JUG', 'B_LSWR', 'B_LSWV', 'B_PLAN', 'B_PLAO', 'B_REGI', 'B_TIMV', 'B_VERN', 'I_DENN', 'I_JUDP', 'I_JUG', 'I_LSWR', 'I_LSWV', 'I_PLAN', 'I_PLAO', 'I_REGI', 'I_TIMV', 'I_VERN', 'O']


### Convert Bentuk Label

## Cleaning Dataset

In [ ]:
def cleaning_text(dataset):
  for i, row in tqdm.tqdm(dataset.iterrows(), total=len(dataset), desc="Cleaning text"):
    dataset.at[i, "text"] = [word.replace("\ufeff", "") for word in row["text"]]
    data_text, data_tag = zip(*[(token, label) for token, label in zip(row["text"], row["label"]) if token.strip() != ''])
    dataset.at[i, "text"] = list(data_text)
    dataset.at[i, "label"] = list(data_tag)

  return dataset

cleaning_text(df)

Cleaning text: 100%|██████████| 200/200 [00:18<00:00, 10.90it/s]


,text,label
0,"[putusan, nomor, 698, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
1,"[penetapan, nomor, 306, /, pdt, ., g, /, 2022,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,"[putusan, nomor, 643, /, pdt, ., g, /, 2022, /...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,"[putusan, nomor, 534, /, pdt, ., g, /, 2020, /...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,"[putusan, nomor, 818, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
...,...,...
195,"[penetapan, nomor, ., 787, /, pdt, ., g, /, 20...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
196,"[putusan, sela, nomor, 190, /, pdt, ., g, /, 2...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
197,"[putusan, nomor, 52, /, pdt, g, /, 2023, /, pn...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
198,"[penetapan, nomor, 680, /, pdt, ., g, /, 2023,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."


## Split token

In [ ]:
def split_token(dataset):
  for i, row in tqdm.tqdm(dataset.iterrows(), total=len(dataset), desc="Processing rows"):
    for index, (text, tag) in enumerate(zip(row["text"], row["label"])):
      if tag != "O":
        token = re.findall(r'[^\/,\.():;]+|[\/\.,():;]', text)
        add_label = []
        for i in range(len(token)):
          if 'B' in tag and not add_label:
            add_label.append(tag)
          else:
            if token[i] != ';':
              add_label.append(tag.replace('B', 'I'))
            else:
              add_label.append('O')

        del row["text"][index]
        del row["label"][index]
        row["text"][index:index] = token
        row["label"][index:index] = add_label
      else:
        token = re.findall(r'[^\/,\.():;]+|[\/\.,():;]', text)
        add_label = []
        for i in range(len(token)):
          add_label.append('O')

        del row["text"][index]
        del row["label"][index]
        row["text"][index:index] = token
        row["label"][index:index] = add_label
  return dataset
split_token(df)

Processing rows: 100%|██████████| 200/200 [01:20<00:00,  2.48it/s]


,text,label
0,"[putusan, nomor, 698, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
1,"[penetapan, nomor, 306, /, pdt, ., g, /, 2022,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,"[putusan, nomor, 643, /, pdt, ., g, /, 2022, /...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
3,"[putusan, nomor, 534, /, pdt, ., g, /, 2020, /...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
4,"[putusan, nomor, 818, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."
...,...,...
195,"[penetapan, nomor, ., 787, /, pdt, ., g, /, 20...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE..."
196,"[putusan, sela, nomor, 190, /, pdt, ., g, /, 2...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
197,"[putusan, nomor, 52, /, pdt, g, /, 2023, /, pn...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
198,"[penetapan, nomor, 680, /, pdt, ., g, /, 2023,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,..."


## Menambahkan kolom Doc

In [ ]:
doc_column = ['doc: ' + str(i + 1) for i in range(len(df))]
df['doc'] = doc_column
df

,text,label,doc
0,"[putusan, nomor, 698, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1
1,"[penetapan, nomor, 306, /, pdt, ., g, /, 2022,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",doc: 2
2,"[putusan, nomor, 643, /, pdt, ., g, /, 2022, /...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",doc: 3
3,"[putusan, nomor, 534, /, pdt, ., g, /, 2020, /...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",doc: 4
4,"[putusan, nomor, 818, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 5
...,...,...,...
195,"[penetapan, nomor, ., 787, /, pdt, ., g, /, 20...","[O, O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VE...",doc: 196
196,"[putusan, sela, nomor, 190, /, pdt, ., g, /, 2...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",doc: 197
197,"[putusan, nomor, 52, /, pdt, g, /, 2023, /, pn...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",doc: 198
198,"[penetapan, nomor, 680, /, pdt, ., g, /, 2023,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 199


## Split data per Sentence

In [ ]:
new_rows = []
for index, row in df.iterrows():
    sentences = ' '.join(row['text']).split(';')
    for sentence in sentences:
        new_rows.append({
            'text': sentence.split(),
            'label': row['label'],
            'doc': row['doc']
        })
new_df = pd.DataFrame(new_rows)
new_df

,text,label,doc
0,"[putusan, nomor, 698, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1
1,"[lawan, :, 01, ., dewan, pengurus, pusat, part...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1
2,"[02, ., drs, ., saur, hutabarat, ,, dr, ., waw...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1
3,"[03, ., dewan, pengurus, wilayah, partai, nasd...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1
4,"[pengadilan, negeri, tersebut]","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1
...,...,...,...
25155,"[dalam, eksepsi, :, menolak, seluruh, eksepsi,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 200
25156,"[dalam, pokok, perkara, :, 01, ., menolak, gug...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 200
25157,"[02, ., menghukum, penggugat, untuk, membayar,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 200
25158,"[demikian, diputuskan, dalam, sidang, permusya...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 200


## Menambahkan kolom Sentence

In [ ]:
new_df['sentence'] = new_df.index.map(lambda x: f'sentence: {x + 1:06}')
new_df

,text,label,doc,sentence
0,"[putusan, nomor, 698, /, pdt, ., g, /, 2021, /...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1,sentence: 000001
1,"[lawan, :, 01, ., dewan, pengurus, pusat, part...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1,sentence: 000002
2,"[02, ., drs, ., saur, hutabarat, ,, dr, ., waw...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1,sentence: 000003
3,"[03, ., dewan, pengurus, wilayah, partai, nasd...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1,sentence: 000004
4,"[pengadilan, negeri, tersebut]","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 1,sentence: 000005
...,...,...,...,...
25155,"[dalam, eksepsi, :, menolak, seluruh, eksepsi,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 200,sentence: 025156
25156,"[dalam, pokok, perkara, :, 01, ., menolak, gug...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 200,sentence: 025157
25157,"[02, ., menghukum, penggugat, untuk, membayar,...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 200,sentence: 025158
25158,"[demikian, diputuskan, dalam, sidang, permusya...","[O, O, B_VERN, I_VERN, I_VERN, I_VERN, I_VERN,...",doc: 200,sentence: 025159


## Tokenisasi per kata

In [ ]:
tokens = []
tags = []
docs = []
sentences = []
for i, row in new_df.iterrows():
    for token, tag in zip(row['text'], row['label']):
        tokens.append(token)
        tags.append(tag)
        docs.append(row['doc'])
        sentences.append(row['sentence'])
tokenized_df = pd.DataFrame({
    'text': tokens,
    'label': tags,
    'doc': docs,
    'sentence': sentences
})
tokenized_df

,text,label,doc,sentence
0,putusan,O,doc: 1,sentence: 000001
1,nomor,O,doc: 1,sentence: 000001
2,698,B_VERN,doc: 1,sentence: 000001
3,/,I_VERN,doc: 1,sentence: 000001
4,pdt,I_VERN,doc: 1,sentence: 000001
...,...,...,...,...
2568225,2023,O,doc: 200,sentence: 025160
2568226,/,O,doc: 200,sentence: 025160
2568227,pn,O,doc: 200,sentence: 025160
2568228,.,O,doc: 200,sentence: 025160


## Menambahkan Kolom Prev dan Next

In [ ]:
tokenized_df['prev'] = tokenized_df['text'].shift(1)
tokenized_df['next'] = tokenized_df['text'].shift(-1)
tokenized_df['prev'].fillna('.', inplace=True)
tokenized_df['next'].fillna('.', inplace=True)

In [ ]:
tokenized_df

,text,label,doc,sentence,prev,next
0,putusan,O,doc: 1,sentence: 000001,.,nomor
1,nomor,O,doc: 1,sentence: 000001,putusan,698
2,698,B_VERN,doc: 1,sentence: 000001,nomor,/
3,/,I_VERN,doc: 1,sentence: 000001,698,pdt
4,pdt,I_VERN,doc: 1,sentence: 000001,/,.
...,...,...,...,...,...,...
2568225,2023,O,doc: 200,sentence: 025160,/,/
2568226,/,O,doc: 200,sentence: 025160,2023,pn
2568227,pn,O,doc: 200,sentence: 025160,/,.
2568228,.,O,doc: 200,sentence: 025160,pn,sby


In [ ]:
proces_data = tokenized_df.rename(columns={'text': 'word', 'label': 'tag'}).reindex(columns=['doc','sentence', 'word', 'prev', 'next', 'tag'])
proces_data

,doc,sentence,word,prev,next,tag
0,doc: 1,sentence: 000001,putusan,.,nomor,O
1,doc: 1,sentence: 000001,nomor,putusan,698,O
2,doc: 1,sentence: 000001,698,nomor,/,B_VERN
3,doc: 1,sentence: 000001,/,698,pdt,I_VERN
4,doc: 1,sentence: 000001,pdt,/,.,I_VERN
...,...,...,...,...,...,...
2568225,doc: 200,sentence: 025160,2023,/,/,O
2568226,doc: 200,sentence: 025160,/,2023,pn,O
2568227,doc: 200,sentence: 025160,pn,/,.,O
2568228,doc: 200,sentence: 025160,.,pn,sby,O


In [ ]:
proces_data.to_csv('Dataset-Perdata200.csv', index=False)